# OVD-Diagnose — Domain 2: Medical (VinDr chest X-ray)

Second specialized domain for the cross-domain fingerprint. VinDr = 14 finding classes with
boxes → exercises the semantic-confusion axis (unlike single-class fish/fruit detection).
Expected contrast vs aerial: subtle low-contrast findings + weak CLIP semantics for medical
terms → likely **L / C-bound** rather than aerial's S-bound.

**Data:** Kaggle → Add Data → a VinBigData chest X-ray dataset that provides **resized PNGs +
a train.csv with boxes** (e.g. a 1024px resized-PNG community dataset). Set the paths below to
the mounted `/kaggle/input/<slug>/...`. **VERIFY** the mount path and that box coords match the
PNG resolution (else use `--scale_from_dims`).

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true
!pip install -q ultralytics transformers pycocotools pyyaml pillow

In [ ]:
# Locate the mounted VinBigData PNG dataset. Adjust to your Add-Data slug.
import glob
print('inputs:', glob.glob('/kaggle/input/*'))
# expect something like /kaggle/input/<slug>/train.csv + a folder of .png images

In [ ]:
# EDIT these two to match your mount, then convert train.csv -> COCO json.
CSV  = '/kaggle/input/CHANGE-ME/train.csv'
IMGS = '/kaggle/input/CHANGE-ME/train'          # dir of .png images
OUT  = 'data/medical/annotations/instances_val.json'

# If your PNGs are resized but boxes are in original DICOM pixels, also build a
# dims csv (image_id,orig_w,orig_h,new_w,new_h) and add --scale_from_dims dims.csv
!python tools/vindr_to_coco.py --csv "{CSV}" --imgs "{IMGS}" --out "{OUT}" --img_ext .png

In [ ]:
# Run all models on the medical domain (same driver as aerial)
LIMIT = 200   # 0 = full; 200 for a first pass
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python tools/run_all.py \
  --ann  data/medical/annotations/instances_val.json \
  --imgs "{IMGS}" \
  --out  runs/diag/medical \
  --device cuda:0 {limit_arg} \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
import pandas as pd
df = pd.read_csv('runs/diag/medical/fingerprints.csv')
print(df.to_string(index=False))

**Cross-domain read:** compare this table to `runs/diag/aerial/fingerprints.csv`. If aerial is
S-bound (high `S_norm`) and medical is L/C-bound (high `L`/`C_ece`, lower `S_norm`), that
divergence is the paper's core figure. If medical is *also* S-bound, that's a capability/vocab
trend across domains — still a finding, just a different one.